# Week 4 Final — PhoBERT v2 + LLM Cascade

Notebook này chỉ giữ luồng final để chạy trên Kaggle:
1. Setup môi trường
2. Pull code mới nhất từ GitHub
3. Load baseline PhoBERT v2 tốt nhất
4. Smoke test cascade trên 10 review
5. Chạy full cascade trên test set


In [ ]:
# Cell 1 — Kaggle GPU + dependencies
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu} | VRAM: {vram:.1f} GB')
else:
    raise RuntimeError('Khong co GPU. Kaggle: Settings -> Accelerator -> GPU')

torch.cuda.empty_cache()
print(f'PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}')

!pip install -q --upgrade pip
!pip install -q transformers==4.41.2 sentence-transformers==2.7.0 tokenizers==0.19.1 accelerate==0.30.1 underthesea py_vncorenlp tabulate tqdm scikit-learn sentencepiece faiss-cpu
!pip install -q openai google-generativeai
print('Dependencies installed')
print('If Kaggle still keeps old packages in memory, restart session and run all cells again.')


In [ ]:
# Cell 2 — Clone/pull latest repo
import os, sys

REPO_URL = 'https://github.com/vudinhminh08/NLP-project-master-study.git'
REPO_BRANCH = 'master'
PROJECT_DIR = '/kaggle/working/absa-project'

if not os.path.exists(PROJECT_DIR):
    !git clone --branch {REPO_BRANCH} --depth=1 {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git pull origin {REPO_BRANCH}

os.chdir(PROJECT_DIR)
print(f'Working dir: {os.getcwd()}')

for p in ['code/week1', 'code/week2', 'code/week3', 'code/week3_part2']:
    if p not in sys.path:
        sys.path.insert(0, p)

for f in [
    'data/train_preprocessed.csv',
    'data/dev_preprocessed.csv',
    'data/test_preprocessed.csv',
    'outputs/eda/class_weights.json',
    'outputs/results/week2_results_VNcoreNLP/models_cls_only/best_model.pt',
    'outputs/results/week2_results_VNcoreNLP/results_cls_only/week2_test_metrics.json',
]:
    print(f'  [{"OK" if os.path.exists(f) else "MISSING"}] {f}')


In [ ]:
# Cell 3 — Load OpenAI API key from Kaggle Secrets
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
OPENAI_API_KEY = secrets.get_secret('OPENAI_API_KEY')
API_KEY = OPENAI_API_KEY
print('OPENAI_API_KEY loaded from Kaggle Secrets')


In [ ]:
# Cell 4 — Verify final Week 4 files
import os, shutil
from glob import glob

KAGGLE_CHECKPOINT_DATASET = 'best_model_pt_phobertv2'
CHECKPOINT_PATH = 'outputs/results/week2_results_VNcoreNLP/models_cls_only/best_model.pt'
EMBEDDINGS_CACHE_PATH = 'outputs/results/embeddings_cache.npy'

def copy_first_match(patterns, dst_path, required=False):
    if os.path.exists(dst_path):
        return dst_path
    for pattern in patterns:
        matches = glob(pattern, recursive=True)
        if matches:
            src_path = matches[0]
            os.makedirs(os.path.dirname(dst_path), exist_ok=True)
            shutil.copy(src_path, dst_path)
            print(f'[COPIED] {src_path} -> {dst_path}')
            return dst_path
    if required:
        print(f'[MISSING] Required file not found for {dst_path}')
    return None

copy_first_match(
    patterns=[
        f'/kaggle/input/{KAGGLE_CHECKPOINT_DATASET}/best_model.pt',
        f'/kaggle/input/{KAGGLE_CHECKPOINT_DATASET}/**/best_model.pt',
        '/kaggle/input/**/best_model.pt',
        '/kaggle/input/**/models_cls_only/best_model.pt',
    ],
    dst_path=CHECKPOINT_PATH,
    required=True,
)

copy_first_match(
    patterns=[
        '/kaggle/input/**/embeddings_cache.npy',
    ],
    dst_path=EMBEDDINGS_CACHE_PATH,
    required=False,
)

required_files = [
    'data/train_preprocessed.csv',
    'data/dev_preprocessed.csv',
    'data/test_preprocessed.csv',
    'outputs/eda/class_weights.json',
    CHECKPOINT_PATH,
    'outputs/results/week2_results_VNcoreNLP/results_cls_only/week2_test_metrics.json',
]

for f in required_files:
    exists = os.path.exists(f)
    size = os.path.getsize(f) if exists else 0
    status = 'OK' if exists else 'MISSING'
    print(f'[{status}] {f} ({size:,} bytes)')

missing = [f for f in required_files if not os.path.exists(f)]
assert not missing, f'Missing files for final v2 cascade run: {missing}'
print('All final-run files verified.')


In [ ]:
# Cell 5 — Setup PhoBERT v2 cascade
import os, sys, json, torch, inspect, importlib
import pandas as pd
from transformers import AutoTokenizer

REPO_ROOT = os.getcwd()
for p in ['code/week1', 'code/week2', 'code/week3', 'code/week3_part2']:
    full = os.path.join(REPO_ROOT, p)
    if full not in sys.path:
        sys.path.insert(0, full)

from utils.constants import PHOBERT_V2, TRAIN_CONFIG, WEAK_ASPECTS, ZERO_TRAIN_ASPECTS
from utils.helpers import set_seed, save_json
from step4_eval import evaluate_predictions
from model import ABSAPhoBERT
from predict import load_best_model
from rag_retriever import ABSARetriever
from llm_client import LLMClient

if 'cascade_predictor' in sys.modules:
    del sys.modules['cascade_predictor']
import cascade_predictor as cp
cp = importlib.reload(cp)

sig = inspect.signature(cp.run_cascade_on_dataset)
print(f'cascade_predictor loaded from: {cp.__file__}')
print(f'run_cascade_on_dataset signature: {sig}')

run_cascade_on_dataset = cp.run_cascade_on_dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
set_seed(TRAIN_CONFIG['seed'])

BEST_MODEL_NAME = PHOBERT_V2
BEST_ENCODER = 'cls_only'
BEST_CHECKPOINT = CHECKPOINT_PATH
BEST_RESULTS_DIR = 'outputs/results/week2_results_VNcoreNLP'
CASCADE_THRESHOLD = 0.60
ABSENT_THRESHOLD = 0.90
MIN_NON_ABSENT_PROB = 0.12
MARGIN_THRESHOLD = 0.15
OVERRIDE_ONLY_FROM_ABSENT = True
CASCADE_K = 4
V2_TEST_METRICS = 'outputs/results/week2_results_VNcoreNLP/results_cls_only/week2_test_metrics.json'
v2_metrics = json.load(open(V2_TEST_METRICS, encoding='utf-8'))

print(f'REPO_ROOT: {REPO_ROOT}')
print(f'Device: {device}')
print(f'Base model: {BEST_MODEL_NAME}')
print(f'Checkpoint: {BEST_CHECKPOINT}')
print(f'Embeddings cache: {EMBEDDINGS_CACHE_PATH} | exists={os.path.exists(EMBEDDINGS_CACHE_PATH)}')
print(f'Cascade params: threshold={CASCADE_THRESHOLD}, absent_threshold={ABSENT_THRESHOLD}, min_non_absent_prob={MIN_NON_ABSENT_PROB}, margin_threshold={MARGIN_THRESHOLD}, override_only_from_absent={OVERRIDE_ONLY_FROM_ABSENT}, k={CASCADE_K}')
print(f'Baseline Combined F1: {v2_metrics["macro_combined_f1"]:.4f}')
print(f'WEAK_ASPECTS ({len(WEAK_ASPECTS)}): {WEAK_ASPECTS}')


In [ ]:
# Cell 6 — Smoke test cascade on 10 reviews
BEST_TOKENIZER = AutoTokenizer.from_pretrained(BEST_MODEL_NAME)
BEST_MODEL = ABSAPhoBERT(
    model_name=BEST_MODEL_NAME,
    dropout=TRAIN_CONFIG['dropout'],
    encoder_option=BEST_ENCODER,
).to(device)
BEST_MODEL = load_best_model(BEST_CHECKPOINT, BEST_MODEL, device)

train_df = pd.read_csv('data/train_preprocessed.csv')
test_df = pd.read_csv('data/test_preprocessed.csv')
retriever = ABSARetriever(cache_path=EMBEDDINGS_CACHE_PATH, use_faiss=True)
retriever.fit(train_df)

api_key = os.environ.get('OPENAI_API_KEY', API_KEY if 'API_KEY' in globals() else '')
if not api_key:
    raise ValueError('Missing OPENAI_API_KEY / API_KEY for cascade.')
llm_client = LLMClient(provider='openai', api_key=api_key)

SMOKE_DIR = os.path.join(BEST_RESULTS_DIR, 'cascade_k4_smoke')
os.makedirs(SMOKE_DIR, exist_ok=True)

y_true_smoke, y_pred_smoke, smoke_stats = run_cascade_on_dataset(
    test_df=test_df,
    train_df=train_df,
    model=BEST_MODEL,
    tokenizer=BEST_TOKENIZER,
    device=device,
    retriever=retriever,
    llm_client=llm_client,
    threshold=CASCADE_THRESHOLD,
    absent_threshold=ABSENT_THRESHOLD,
    min_non_absent_prob=MIN_NON_ABSENT_PROB,
    margin_threshold=MARGIN_THRESHOLD,
    override_only_from_absent=OVERRIDE_ONLY_FROM_ABSENT,
    k=CASCADE_K,
    sleep_sec=0.0,
    max_samples=10,
    return_records=True,
)

smoke_metrics = evaluate_predictions(
    y_true_smoke,
    y_pred_smoke,
    title='Cascade Smoke Test (10 reviews) — PhoBERT v2',
    exclude_aspects=ZERO_TRAIN_ASPECTS,
    save_path=os.path.join(SMOKE_DIR, 'cascade_smoke_metrics.json'),
)
save_json(smoke_stats, os.path.join(SMOKE_DIR, 'cascade_smoke_stats.json'))

print(f'Base model for cascade: {BEST_MODEL_NAME}')
print(json.dumps(smoke_stats, ensure_ascii=False, indent=2))
print(f"Smoke Combined F1: {smoke_metrics['macro_combined_f1']:.4f}")


In [ ]:
# Cell 7 — Full cascade on test set
CASCADE_DIR = os.path.join(BEST_RESULTS_DIR, 'cascade_k4')
os.makedirs(CASCADE_DIR, exist_ok=True)

y_true_cascade, y_pred_cascade, cascade_stats = run_cascade_on_dataset(
    test_df=test_df,
    train_df=train_df,
    model=BEST_MODEL,
    tokenizer=BEST_TOKENIZER,
    device=device,
    retriever=retriever,
    llm_client=llm_client,
    threshold=CASCADE_THRESHOLD,
    absent_threshold=ABSENT_THRESHOLD,
    min_non_absent_prob=MIN_NON_ABSENT_PROB,
    margin_threshold=MARGIN_THRESHOLD,
    override_only_from_absent=OVERRIDE_ONLY_FROM_ABSENT,
    k=CASCADE_K,
    sleep_sec=1.0,
    return_records=True,
)

cascade_metrics = evaluate_predictions(
    y_true_cascade,
    y_pred_cascade,
    title='Cascade Test — PhoBERT v2',
    exclude_aspects=ZERO_TRAIN_ASPECTS,
    save_path=os.path.join(CASCADE_DIR, 'cascade_test_metrics.json'),
)
save_json(cascade_stats, os.path.join(CASCADE_DIR, 'cascade_stats.json'))

comparison = {
    'baseline_v2_cls_only': {
        'model': BEST_MODEL_NAME,
        'combined_f1': v2_metrics['macro_combined_f1'],
        'acd_f1': v2_metrics['macro_acd_f1'],
        'spc_f1': v2_metrics['macro_spc_f1'],
    },
    'cascade_v2_rag_k4': {
        'model': BEST_MODEL_NAME,
        'combined_f1': cascade_metrics['macro_combined_f1'],
        'acd_f1': cascade_metrics['macro_acd_f1'],
        'spc_f1': cascade_metrics['macro_spc_f1'],
        'trigger_rate': cascade_stats['trigger_rate'],
        'total_overrides': cascade_stats['total_overrides'],
    },
}
save_json(comparison, os.path.join(CASCADE_DIR, 'baseline_vs_cascade_comparison.json'))

print(f'WEAK_ASPECTS used: {WEAK_ASPECTS}')
print(json.dumps(cascade_stats, ensure_ascii=False, indent=2))
print(json.dumps(comparison, ensure_ascii=False, indent=2))
print(f"Cascade Combined F1: {cascade_metrics['macro_combined_f1']:.4f}")


In [ ]:
# Cell T1 — Setup dataloaders + collect logits for threshold tuning
import os, sys, json
from step2_dataloader import create_dataloaders
from train import load_class_weights
from threshold_tuner import collect_logits, logits_to_probs

THRESHOLD_DIR = os.path.join(BEST_RESULTS_DIR, 'threshold_tuning')
os.makedirs(THRESHOLD_DIR, exist_ok=True)

if 'BEST_TOKENIZER' not in globals():
    BEST_TOKENIZER = AutoTokenizer.from_pretrained(BEST_MODEL_NAME)
if 'BEST_MODEL' not in globals():
    BEST_MODEL = ABSAPhoBERT(
        model_name=BEST_MODEL_NAME,
        dropout=TRAIN_CONFIG['dropout'],
        encoder_option=BEST_ENCODER,
    ).to(device)
    BEST_MODEL = load_best_model(BEST_CHECKPOINT, BEST_MODEL, device)

train_loader_t, dev_loader_t, test_loader_t = create_dataloaders(
    train_path='data/train_preprocessed.csv',
    dev_path='data/dev_preprocessed.csv',
    test_path='data/test_preprocessed.csv',
    tokenizer=BEST_TOKENIZER,
    batch_size=TRAIN_CONFIG['batch_size'],
    max_len=TRAIN_CONFIG['max_seq_len'],
    num_workers=2,
    use_preprocessed=True,
)
CLASS_WEIGHTS = load_class_weights(
    'outputs/eda/class_weights.json',
    weight_clip=TRAIN_CONFIG['weight_clip'],
    device=device,
)

dev_logits_raw, y_dev_true = collect_logits(BEST_MODEL, dev_loader_t, device)
test_logits_raw, y_test_true = collect_logits(BEST_MODEL, test_loader_t, device)
dev_probs = logits_to_probs(dev_logits_raw)
test_probs = logits_to_probs(test_logits_raw)

print(f'dev_logits_raw shape: {dev_logits_raw.shape}')
print(f'test_logits_raw shape: {test_logits_raw.shape}')
print(f'y_dev_true shape: {y_dev_true.shape}')
print(f'y_test_true shape: {y_test_true.shape}')
print(f'THRESHOLD_DIR: {THRESHOLD_DIR}')


In [ ]:
# Cell T2 — Tune per-aspect thresholds on dev
from threshold_tuner import find_best_thresholds, apply_thresholds
from utils.constants import ASPECT_COLUMNS, ZERO_TRAIN_ASPECTS
from utils.helpers import save_json

thresholds = find_best_thresholds(
    dev_probs,
    y_dev_true,
    aspect_columns=ASPECT_COLUMNS,
    exclude_aspects=ZERO_TRAIN_ASPECTS,
)
save_json(thresholds, os.path.join(THRESHOLD_DIR, 'per_aspect_thresholds.json'))

threshold_rows = []
for aspect, cfg in thresholds.items():
    threshold_rows.append({
        'aspect': aspect,
        'use_argmax': cfg.get('use_argmax', False),
        'threshold': cfg.get('threshold'),
        'support': cfg.get('support'),
        'baseline_combined_f1': cfg.get('baseline_combined_f1'),
        'dev_combined_f1': cfg.get('dev_combined_f1'),
        'improvement': cfg.get('improvement'),
    })

threshold_df = pd.DataFrame(threshold_rows).sort_values(
    by=['use_argmax', 'improvement', 'support'], ascending=[True, False, False]
)
display(threshold_df)
print('Num tuned aspects:', int((~threshold_df['use_argmax']).sum()))


In [ ]:
# Cell T3 — Apply tuned thresholds on test + evaluate
from step4_eval import evaluate_predictions
from utils.helpers import save_json

y_dev_argmax = dev_probs.argmax(axis=-1)
y_dev_tuned = apply_thresholds(dev_probs, thresholds, aspect_columns=ASPECT_COLUMNS)
y_test_argmax = test_probs.argmax(axis=-1)
y_test_tuned = apply_thresholds(test_probs, thresholds, aspect_columns=ASPECT_COLUMNS)

dev_argmax_metrics = evaluate_predictions(
    y_dev_true, y_dev_argmax,
    title='Dev — Argmax Baseline',
    exclude_aspects=ZERO_TRAIN_ASPECTS,
    save_path=os.path.join(THRESHOLD_DIR, 'dev_argmax_metrics.json'),
)
dev_tuned_metrics = evaluate_predictions(
    y_dev_true, y_dev_tuned,
    title='Dev — Threshold Tuned',
    exclude_aspects=ZERO_TRAIN_ASPECTS,
    save_path=os.path.join(THRESHOLD_DIR, 'dev_threshold_tuned_metrics.json'),
)
test_argmax_metrics = evaluate_predictions(
    y_test_true, y_test_argmax,
    title='Test — Argmax Baseline',
    exclude_aspects=ZERO_TRAIN_ASPECTS,
    save_path=os.path.join(THRESHOLD_DIR, 'test_argmax_metrics.json'),
)
test_tuned_metrics = evaluate_predictions(
    y_test_true, y_test_tuned,
    title='Test — Threshold Tuned',
    exclude_aspects=ZERO_TRAIN_ASPECTS,
    save_path=os.path.join(THRESHOLD_DIR, 'test_threshold_tuned_metrics.json'),
)

threshold_comparison = {
    'dev': {
        'argmax_combined_f1': dev_argmax_metrics['macro_combined_f1'],
        'threshold_tuned_combined_f1': dev_tuned_metrics['macro_combined_f1'],
        'improvement': dev_tuned_metrics['macro_combined_f1'] - dev_argmax_metrics['macro_combined_f1'],
    },
    'test': {
        'argmax_combined_f1': test_argmax_metrics['macro_combined_f1'],
        'threshold_tuned_combined_f1': test_tuned_metrics['macro_combined_f1'],
        'improvement': test_tuned_metrics['macro_combined_f1'] - test_argmax_metrics['macro_combined_f1'],
    },
}
save_json(threshold_comparison, os.path.join(THRESHOLD_DIR, 'threshold_tuning_summary.json'))
print(json.dumps(threshold_comparison, ensure_ascii=False, indent=2))


In [ ]:
# Cell T4 — Overall comparison table
summary_rows = [
    {
        'method': 'PhoBERT v2 argmax',
        'combined_f1': test_argmax_metrics['macro_combined_f1'],
        'acd_f1': test_argmax_metrics['macro_acd_f1'],
        'spc_f1': test_argmax_metrics['macro_spc_f1'],
    },
    {
        'method': 'PhoBERT v2 threshold-tuned',
        'combined_f1': test_tuned_metrics['macro_combined_f1'],
        'acd_f1': test_tuned_metrics['macro_acd_f1'],
        'spc_f1': test_tuned_metrics['macro_spc_f1'],
    },
    {
        'method': 'PhoBERT v2 ensemble + threshold (previous)',
        'combined_f1': 0.5644,
        'acd_f1': None,
        'spc_f1': None,
    },
    {
        'method': 'Cascade v1.5 (previous best hybrid)',
        'combined_f1': 0.5639,
        'acd_f1': None,
        'spc_f1': None,
    },
]
summary_df = pd.DataFrame(summary_rows).sort_values('combined_f1', ascending=False).reset_index(drop=True)
display(summary_df)
best_row = summary_df.iloc[0]
print(f"Best method: {best_row['method']} | Combined F1 = {best_row['combined_f1']:.4f}")
